# AbRank Co-Folded 3D: Quickstart & Multi-Method Pose Explorer

This notebook walks through the **34,040 template-free wild-type (WT) antibody–antigen and nanobody–antigen complexes** hosted in [`aurlienplissier/abrank-cofolded-3d`](https://www.kaggle.com/datasets/aurlienplissier/abrank-cofolded-3d) (`851` WT scaffolds × `4` co-folding models × `10` diffusion poses), alongside the main [`aurlienplissier/abrank`](https://www.kaggle.com/datasets/aurlienplissier/abrank) v2 benchmark.

### What you will learn
1. **Explore the 851 WT Parental Scaffolds** (`cofolding_manifest.tsv`): priority tiers (`1_loaco_local`, `2_candidate_and_vhh`, `3_cross_modal_and_global`), experimental RCSB co-crystals (`340`) vs. unresolved targets (`511`), and paired IgG/Fab (`3` chains) vs. single-domain VHH nanobodies (`2` chains).
2. **Audit the 34,040-Pose Inventory** (`MANIFEST_structures.tsv` & `MANIFEST_confidence.tsv`) across **AlphaFold 3**, **Boltz-2**, **Chai-1**, and **Protenix**.
3. **Stream Structures & Confidence Scores Directly from `.tgz.bin` Archives** via Python `tarfile.open(..., 'r:gz')` without unpacking gigabytes to disk — computing cross-interface **`Ab–Ag Pair ipTM`** (removing `VH–VL` dimer confounding on 3-chain Fabs) and inspecting standardized `(H, [L], A)` `LOACO-Local` PDB coordinates.

Links: [AI-SysBio/AbRank](https://github.com/AI-SysBio/AbRank) · [Main AbRank v2 Dataset](https://www.kaggle.com/datasets/aurlienplissier/abrank) · [AI-SysBio/logica](https://github.com/AI-SysBio/logica/tree/main/antibodies-abrank)


In [ ]:
import json
import tarfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
})


def find_cofolded_root() -> Path:
    candidates = [
        Path('/kaggle/input/abrank-cofolded-3d'),
        Path('/kaggle/input/datasets/aurlienplissier/abrank-cofolded-3d'),
        Path('/tmp/abrank-cofolded-3d-bundle'),
    ]
    for c in candidates:
        if (c / 'cofolding_manifest.tsv').exists():
            return c
    for p in Path('/kaggle/input').rglob('cofolding_manifest.tsv'):
        if (p.parent / 'MANIFEST_structures.tsv').exists():
            return p.parent
    raise FileNotFoundError('Could not locate abrank-cofolded-3d directory.')


ROOT = find_cofolded_root()
print(f'Resolved AbRank Co-Folded 3D root: {ROOT}')
for p in sorted(ROOT.iterdir()):
    if p.is_file():
        print(f'  {p.name:40s}  {p.stat().st_size / 1e6:8.2f} MB')


## 1. Inspect the 851 Wild-Type Parental Scaffolds (`cofolding_manifest.tsv`)

Every parental scaffold — including the `340` complexes that already have experimental RCSB PDB co-crystals — was folded **template-free (`templates=[]`)** across all 4 engines. This allows direct `DockQ` validation against native co-crystals alongside uniform crystal-free inference on the `511` unresolved targets.


In [ ]:
manifest = pd.read_csv(ROOT / 'cofolding_manifest.tsv', sep='\t')
st_manifest = pd.read_csv(ROOT / 'MANIFEST_structures.tsv', sep='\t')
cf_manifest = pd.read_csv(ROOT / 'MANIFEST_confidence.tsv', sep='\t')

print(f'WT parental scaffolds : {len(manifest):,}')
print(f'  Solved co-crystals  : {int(manifest["is_experimental_cocrystal"].sum()):,}')
print(f'  Unresolved targets  : {int((~manifest["is_experimental_cocrystal"]).sum()):,}')
print(f'  Paired VH+VL+Ag (3) : {int((manifest["n_chains"] == 3).sum()):,}')
print(f'  Nanobody VHH+Ag (2) : {int((manifest["n_chains"] == 2).sum()):,}')
print(f'Indexed structure files : {len(st_manifest):,} ({st_manifest["bytes"].sum() / 1e9:.2f} GB uncompressed)')
print(f'Indexed confidence files: {len(cf_manifest):,} ({cf_manifest["bytes"].sum() / 1e6:.1f} MB uncompressed)')

tier_summary = (
    manifest.groupby('priority_tier')
    .agg(
        n_scaffolds=('scaffold_id', 'count'),
        n_cocrystal=('is_experimental_cocrystal', 'sum'),
        n_vhh=('n_chains', lambda s: int((s == 2).sum())),
        median_total_aa=('total_aa_len', 'median'),
        max_total_aa=('total_aa_len', 'max'),
    )
    .reset_index()
)
display(tier_summary)
display(manifest[['scaffold_id', 'priority_tier', 'is_experimental_cocrystal', 'wt_pdb_id', 'n_chains', 'len_H', 'len_L', 'len_A']].head(5))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))

# Panel A: Scaffolds per priority tier split by experimental co-crystal status
tier_ct = pd.crosstab(manifest['priority_tier'], manifest['is_experimental_cocrystal'])
tier_labels = ['Tier 1:\nLOACO-Local', 'Tier 2:\nCandidate & VHH', 'Tier 3:\nCross-Modal & Global']
x = np.arange(len(tier_ct))
b1 = axes[0].bar(x, tier_ct[True], label='Solved Co-Crystal (n=340)', color='#1f77b4')
b2 = axes[0].bar(x, tier_ct[False], bottom=tier_ct[True], label='Unresolved Target (n=511)', color='#ff7f0e')
for i, total in enumerate((tier_ct[True] + tier_ct[False]).values):
    axes[0].text(i, total + 10, f'{int(total)}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(tier_labels, fontsize=9)
axes[0].set_ylabel('WT Parental Scaffolds')
axes[0].set_ylim(0, 520)
axes[0].set_title('A. Priority Tiers & Experimental Coverage')
axes[0].legend(frameon=False, fontsize=8)

# Panel B: Total folded complex sequence length distribution
paired_len = manifest.loc[manifest['n_chains'] == 3, 'total_aa_len']
vhh_len = manifest.loc[manifest['n_chains'] == 2, 'total_aa_len']
axes[1].hist(paired_len, bins=30, alpha=0.75, label=f'Paired VH+VL+Ag (n={len(paired_len)})', color='#2ca02c')
axes[1].hist(vhh_len, bins=25, alpha=0.75, label=f'VHH+Ag (n={len(vhh_len)})', color='#9467bd')
axes[1].set_xlabel('Total Complex Length (Amino Acids)')
axes[1].set_ylabel('Number of Scaffolds')
axes[1].set_title('B. Complex Size Distribution (851 Scaffolds)')
axes[1].legend(frameon=False, fontsize=8)

# Panel C: Per-method uncompressed structure footprint across 8,510 poses each
method_stats = (
    st_manifest.groupby('method')
    .agg(n_poses=('pose_index', 'count'), total_gb=('bytes', lambda s: s.sum() / 1e9), median_kb=('bytes', lambda s: s.median() / 1e3))
    .reindex(['af3', 'boltz2', 'chai1', 'protenix'])
)
colors = ['#1f77b4', '#d62728', '#2ca02c', '#9467bd']
bars = axes[2].bar(method_stats.index, method_stats['total_gb'], color=colors, width=0.55)
for bar, (_, row) in zip(bars, method_stats.iterrows()):
    axes[2].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.08,
        f"{row['total_gb']:.2f} GB\n({int(row['n_poses']):,})",
        ha='center',
        va='bottom',
        fontsize=8.5,
    )
axes[2].set_ylabel('Uncompressed mmCIF Volume (GB)')
axes[2].set_ylim(0, 3.6)
axes[2].set_title('C. 34,040 Co-Folded Poses by Method')

plt.tight_layout()
plt.show()


## 2. Stream a Standardized LOACO-Local 3D Complex Directly from `.tgz.bin`

All five archive files use the `.tgz.bin` extension (standard gzip-compressed tar archives) so Kaggle preserves them cleanly without exploding `71,842` individual files. You can inspect or extract specific poses on the fly with Python's built-in `tarfile.open(..., 'r:gz')` — no renaming or full extraction needed!

Below we stream a standardized `(H, L, A)` complex PDB from `cofolded_standardized_loaco.tgz.bin`, parse the `Cα` backbone coordinates of the antibody (`H`, `L`) and antigen (`A`), and compute the minimum paratope–epitope heavy/light–antigen contact distance map.


In [ ]:
def parse_pdb_ca(pdb_text: str) -> dict[str, np.ndarray]:
    chains: dict[str, list[list[float]]] = {}
    for line in pdb_text.splitlines():
        if line.startswith('ATOM  ') and line[12:16].strip() == 'CA':
            ch = line[21].strip()
            xyz = [float(line[30:38]), float(line[38:46]), float(line[46:54])]
            chains.setdefault(ch, []).append(xyz)
    return {ch: np.asarray(coords, dtype=np.float64) for ch, coords in chains.items()}


std_archive = ROOT / 'cofolded_standardized_loaco.tgz.bin'
sample_name, sample_pdb = None, None
with tarfile.open(std_archive, 'r:gz') as tf:
    for member in tf:
        if member.isfile() and member.name.endswith('__p00.pdb') and '/PDB_' in member.name:
            sample_name = member.name
            sample_pdb = tf.extractfile(member).read().decode('utf-8')
            break

ca = parse_pdb_ca(sample_pdb)
print(f'Streamed standardized complex: {sample_name}')
for ch, coords in ca.items():
    print(f'  Chain {ch}: {len(coords)} C-alpha residues')

ab_ca = np.vstack([ca['H'], ca['L']]) if 'L' in ca else ca['H']
ag_ca = ca['A']
dist_mat = np.linalg.norm(ab_ca[:, None, :] - ag_ca[None, :, :], axis=-1)
n_contacts_85 = int((dist_mat <= 8.5).sum())
min_dist = float(dist_mat.min())
print(f'  Inter-chain Ab-Ag C-alpha contacts (<= 8.5 A): {n_contacts_85}, min distance = {min_dist:.2f} A')

fig, ax = plt.subplots(figsize=(7.5, 4.8))
im = ax.imshow(np.clip(dist_mat, 0, 25.0), aspect='auto', cmap='viridis_r', origin='lower')
if 'L' in ca:
    ax.axhline(len(ca['H']) - 0.5, color='white', linestyle='--', linewidth=1.2, label='VH / VL boundary')
    ax.legend(loc='upper right', frameon=True, fontsize=8.5)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label('C-alpha Distance (Angstroms, clipped at 25 A)')
ax.set_xlabel('Antigen Residue Index (Chain A)')
ax.set_ylabel('Antibody Residue Index (Chains H + L)')
ax.set_title(f'Ab-Ag C-alpha Contact Map: {Path(sample_name).name}')
plt.tight_layout()
plt.show()


## 3. Reference Co-Folding Pose Accuracy (`DockQ`) Across All 340 Native Co-Crystals

Evaluating all `34,040` template-free poses (`13,600` poses on the `340` experimentally solved WT co-crystals) highlights two crucial practical findings for antibody–antigen modeling (full reproducible script in [`scripts/cofolding/03_ingest_and_evaluate_cofolded.py`](https://github.com/AI-SysBio/AbRank/blob/main/scripts/cofolding/03_ingest_and_evaluate_cofolded.py)):
1. **Cross-interface `Ab–Ag Pair ipTM` vs. Global `ipTM`**: On 3-chain Fabs (`VH + VL + Ag`), global `ipTM` is inflated by the tight `VH–VL` dimer interface even when the antigen is mis-docked. Extracting cross-interface **`Ab–Ag Pair ipTM`** yields strong calibration with native `DockQ` across all 4 solvers (Spearman `rho = +0.684` to `+0.771`).
2. **Multi-pose sampling (`K = 10`) & 40-pose pooling** substantially improve pose recovery (`Boltz-2` Top-`ipTM` `DockQ = 0.550` with `65.9% >= 0.23`; `40-Pose Multi-Method Pool` Top-`ipTM` `DockQ = 0.577` and Best-of-40 oracle `DockQ = 0.677` with `77.9% >= 0.23`).


In [ ]:
dockq_summary = pd.DataFrame([
    {'Method': 'AlphaFold 3', 'Expected_1Pose_DockQ': 0.300, 'Top_ipTM_DockQ': 0.340, 'Best_of_10_DockQ': 0.415, 'Pct_Acceptable_Top_ipTM': 43.8, 'Spearman_AbAg_ipTM': 0.684},
    {'Method': 'Protenix',    'Expected_1Pose_DockQ': 0.353, 'Top_ipTM_DockQ': 0.380, 'Best_of_10_DockQ': 0.456, 'Pct_Acceptable_Top_ipTM': 47.4, 'Spearman_AbAg_ipTM': 0.731},
    {'Method': 'Chai-1',      'Expected_1Pose_DockQ': 0.392, 'Top_ipTM_DockQ': 0.395, 'Best_of_10_DockQ': 0.440, 'Pct_Acceptable_Top_ipTM': 50.3, 'Spearman_AbAg_ipTM': 0.771},
    {'Method': 'Boltz-2',     'Expected_1Pose_DockQ': 0.516, 'Top_ipTM_DockQ': 0.550, 'Best_of_10_DockQ': 0.598, 'Pct_Acceptable_Top_ipTM': 65.9, 'Spearman_AbAg_ipTM': 0.731},
    {'Method': '40-Pose Pool','Expected_1Pose_DockQ': np.nan,'Top_ipTM_DockQ': 0.577, 'Best_of_10_DockQ': 0.677, 'Pct_Acceptable_Top_ipTM': 68.8, 'Spearman_AbAg_ipTM': np.nan},
])
display(dockq_summary)

fig, ax = plt.subplots(figsize=(8.8, 4.4))
x = np.arange(len(dockq_summary))
w = 0.26
ax.bar(x - w, dockq_summary['Expected_1Pose_DockQ'].fillna(0), w, label='Expected 1-Pose DockQ', color='#aec7e8')
ax.bar(x,     dockq_summary['Top_ipTM_DockQ'], w, label='Top-ipTM Selected DockQ', color='#1f77b4')
ax.bar(x + w, dockq_summary['Best_of_10_DockQ'], w, label='Best-of-K Oracle DockQ', color='#ff7f0e')
for i, row in dockq_summary.iterrows():
    ax.text(i, row['Top_ipTM_DockQ'] + 0.015, f"{row['Top_ipTM_DockQ']:.3f}", ha='center', va='bottom', fontsize=8.5, fontweight='bold')
    ax.text(i + w, row['Best_of_10_DockQ'] + 0.015, f"{row['Best_of_10_DockQ']:.3f}", ha='center', va='bottom', fontsize=8.5)
ax.set_xticks(x)
ax.set_xticklabels(dockq_summary['Method'])
ax.set_ylabel('Mean DockQ vs. 340 Native Co-Crystals')
ax.set_ylim(0, 0.78)
ax.set_title('Template-Free Multi-Pose DockQ Recovery Across All 340 Solved WT Co-Crystals')
ax.legend(frameon=False, loc='upper left')
plt.tight_layout()
plt.show()


## Summary & Next Steps

### Data Analysis Key Findings
- **Complete 851-Scaffold Coverage**: All `851` AbRank v2 WT scaffolds (`340` experimentally solved RCSB co-crystals + `511` unresolved targets; `566` paired `VH+VL+Ag` complexes and `285` single-domain `VHH+Ag` nanobody complexes; `94` Tier-1 LOACO-Local, `416` Tier-2 Candidate/VHH, `341` Tier-3 Cross-Modal/Global) have `10` template-free poses across **AF3**, **Boltz-2**, **Chai-1**, and **Protenix** (`34,040` mmCIF structures + `34,040` confidence summaries = `10.5+ GB` uncompressed, packaged into `3.1 GB` across 8 clean files).
- **Zero-Friction Streaming & Extraction**: `.tgz.bin` archives stream individual PDB/mmCIF/JSON files directly via `tarfile.open(..., 'r:gz')` or unpack directly into `AI-SysBio/AbRank`'s `data/structures/` tree with `tar -xzf <method>_10poses.tgz.bin -C data/structures/`.
- **Strong 3D Recovery Without Native Co-Crystals**: `Boltz-2` achieves `0.550` Top-`ipTM` mean `DockQ` across all `340` solved co-crystals (`65.9% >= 0.23`), and combining all 4 methods (`40-Pose Pool`) boosts Top-`ipTM` `DockQ` to `0.577` (`68.8% >= 0.23`, Best-of-40 oracle `0.677`).

### Insights or Next Steps
1. **Skip GPU Co-Folding Locally**: Clone [`AI-SysBio/AbRank`](https://github.com/AI-SysBio/AbRank) and run `python scripts/cofolding/00_download_cofolded_from_kaggle.py --methods boltz2 chai1 --include-standardized` (open-weight models only) or `--methods all`.
2. **Run Multi-Pose 3D Baselines**: Score `ProteinMPNN`, `ESM-IF1`, `AntiFold`, `ANTIPASTI`, `GearBind`, or your own 3D model on `cofolded_standardized_loaco.tgz.bin` and evaluate with `python -m abrank.evaluate --suite maturation --kaggle-dataset aurlienplissier/abrank --preds my_preds.tsv`.
3. **Respect AlphaFold 3 Output Terms**: `boltz2`, `chai1`, and `protenix` are open-weight (`MIT`/`Apache-2.0`); `af3_10poses.tgz.bin` is restricted to non-commercial academic research under Google DeepMind's AlphaFold 3 Output Terms.
